# DL02 — Transformer: Attention Is All You Need


## 0. Paper Information
| | |
|---|---|
| **Paper** | Attention Is All You Need |
| **Authors** | Vaswani, A., Shazeer, N., Parmar, N., et al. |
| **Venue** | NeurIPS 2017 |
| **arXiv** | [1706.03762](https://arxiv.org/abs/1706.03762) |
| **Key Contribution** | Self-attention replaces recurrence entirely; parallelizable & superior |
| **This notebook** | Full Transformer from scratch + mini EN→ZH translation demo |

**Why it matters for economics & finance:**
Transformer is the backbone of GPT, BERT, FinBERT, and all modern LLMs used in
financial NLP (sentiment analysis, earnings call parsing, regulatory text mining).
Understanding its internals is essential for adapting pre-trained models to economic text.


---
## 1. Architecture & Key Design Decisions
### 1.1 Encoder–Decoder Structure
```
Input tokens → Embedding + Positional Encoding
  → Encoder (N=6 identical layers):
      Multi-Head Self-Attention → Add & Norm
      Feed-Forward Network     → Add & Norm
  → Decoder (N=6 identical layers):
      Masked Multi-Head Self-Attention → Add & Norm
      Encoder–Decoder Attention        → Add & Norm
      Feed-Forward Network             → Add & Norm
  → Linear + Softmax → Output probabilities
```

### 1.2 Key Hyperparameters (paper defaults)
| Parameter | Value |
|-----------|-------|
| d_model | 512 |
| num_heads | 8 |
| d_ff | 2048 |
| N (layers) | 6 |
| dropout | 0.1 |


---
## 2. Mathematical Foundation
### 2.1 Scaled Dot-Product Attention (Eq. 1)
$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
- Scaling by $\sqrt{d_k}$ prevents dot products from growing large → gradient vanishing in softmax

### 2.2 Multi-Head Attention (Eq. 4–5)
$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1,\ldots,\text{head}_h)W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$
Each head attends to different representation subspaces.

### 2.3 Position-wise FFN (Eq. 2)
$$\text{FFN}(x) = \max(0,\; xW_1+b_1)W_2+b_2$$

### 2.4 Positional Encoding (Eq. 3–4)
$$PE_{(pos,2i)} = \sin(pos/10000^{2i/d_{model}})$$
$$PE_{(pos,2i+1)} = \cos(pos/10000^{2i/d_{model}})$$


---
## 3. Core Implementation


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


### 3.1 Scaled Dot-Product Attention


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Eq. 1 in Vaswani et al. (2017).

    Args:
        Q, K, V: (batch, heads, seq_len, d_k)
        mask: optional boolean mask (True = attend)
    Returns:
        output: (batch, heads, seq_len, d_v)
        attn_weights: (batch, heads, seq_len, seq_len)
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = F.softmax(scores, dim=-1)
    return torch.matmul(attn, V), attn


### 3.2 Multi-Head Attention (Eq. 4–5)


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'
        self.d_k = d_model // num_heads
        self.h   = num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        def split_heads(x):
            return x.view(B, -1, self.h, self.d_k).transpose(1, 2)
        Q = split_heads(self.W_q(q))
        K = split_heads(self.W_k(k))
        V = split_heads(self.W_v(v))
        out, self.attn = scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.h * self.d_k)
        return self.dropout(self.W_o(out))


### 3.3 FFN, Positional Encoding, Encoder/Decoder Layers


In [ ]:
class FeedForward(nn.Module):
    """Eq. 2: FFN(x) = max(0, xW1+b1)W2+b2"""
    def __init__(self, d_model, d_ff=2048, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
    def forward(self, x): return self.net(x)


class PositionalEncoding(nn.Module):
    """Eq. 3-4: sinusoidal positional encoding."""
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn       = FeedForward(d_model, d_ff, dropout)
        self.norm1     = nn.LayerNorm(d_model)
        self.norm2     = nn.LayerNorm(d_model)
        self.drop      = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        x = self.norm1(x + self.drop(self.self_attn(x, x, x, src_mask)))
        x = self.norm2(x + self.drop(self.ffn(x)))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn        = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, memory, tgt_mask=None, mem_mask=None):
        x = self.norm1(x + self.drop(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.drop(self.cross_attn(x, memory, memory, mem_mask)))
        x = self.norm3(x + self.drop(self.ffn(x)))
        return x


### 3.4 Full Transformer


In [ ]:
class Transformer(nn.Module):
    """Full encoder-decoder Transformer (Vaswani et al., 2017)."""
    def __init__(self, src_vocab, tgt_vocab, d_model=512, num_heads=8,
                 num_layers=6, d_ff=2048, dropout=0.1, max_len=512):
        super().__init__()
        self.src_emb = nn.Embedding(src_vocab, d_model)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout, max_len)
        self.scale   = math.sqrt(d_model)
        self.encoder = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout)
                                      for _ in range(num_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout)
                                      for _ in range(num_layers)])
        self.out_proj = nn.Linear(d_model, tgt_vocab)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1: nn.init.xavier_uniform_(p)

    def encode(self, src, src_mask=None):
        x = self.pos_enc(self.src_emb(src) * self.scale)
        for layer in self.encoder: x = layer(x, src_mask)
        return x

    def decode(self, tgt, memory, tgt_mask=None, mem_mask=None):
        x = self.pos_enc(self.tgt_emb(tgt) * self.scale)
        for layer in self.decoder: x = layer(x, memory, tgt_mask, mem_mask)
        return x

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        memory = self.encode(src, src_mask)
        out    = self.decode(tgt, memory, tgt_mask, src_mask)
        return self.out_proj(out)


def make_causal_mask(sz, device):
    """Upper-triangular mask for autoregressive decoding."""
    return torch.tril(torch.ones(sz, sz, device=device)).bool()


# Quick sanity check
model = Transformer(src_vocab=1000, tgt_vocab=1000, d_model=128, num_heads=4,
                    num_layers=2, d_ff=256).to(device)
src = torch.randint(0, 1000, (2, 10)).to(device)
tgt = torch.randint(0, 1000, (2, 8)).to(device)
mask = make_causal_mask(8, device)
out = model(src, tgt, tgt_mask=mask)
print('Output shape:', out.shape)  # (2, 8, 1000)
total = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total/1e3:.1f}K (tiny demo model)')
print(f'Full paper model (d=512, h=8, N=6): ~65M parameters')


---
## 4. Training Pipeline — Mini EN→ZH Translation


In [ ]:
# ── Minimal bilingual dataset ─────────────────────────────────────────────
en_sentences = [
    'i love you', 'he likes music', 'she eats apple',
    'we study math', 'they play football', 'i read book',
    'he drinks water', 'she runs fast', 'we watch movie', 'they sing song'
]
zh_sentences = [
    '我 爱 你', '他 喜欢 音乐', '她 吃 苹果',
    '我们 学习 数学', '他们 踢 足球', '我 读书',
    '他 喝水', '她 跑 很快', '我们 看 电影', '他们 唱歌'
]


class Vocab:
    PAD, BOS, EOS = 0, 1, 2
    def __init__(self, sentences, chinese=False):
        words = set()
        for s in sentences:
            words.update(list(s) if chinese else s.split())
        self.w2i = {'<PAD>':0,'<BOS>':1,'<EOS>':2}
        self.w2i.update({w: i+3 for i,w in enumerate(sorted(words))})
        self.i2w = {v:k for k,v in self.w2i.items()}

    def encode(self, sentence, max_len, chinese=False):
        toks = list(sentence) if chinese else sentence.split()
        ids = [self.w2i.get(t, 0) for t in toks]
        ids = [self.BOS] + ids + [self.EOS]
        ids = ids[:max_len] + [self.PAD] * max(0, max_len - len(ids))
        return torch.tensor(ids, dtype=torch.long)

    def decode(self, ids):
        return [self.i2w.get(i,'?') for i in ids if i not in (self.PAD, self.BOS, self.EOS)]


MAX_EN, MAX_ZH = 7, 8
en_vocab = Vocab(en_sentences)
zh_vocab = Vocab(zh_sentences, chinese=True)

src_data = torch.stack([en_vocab.encode(s, MAX_EN) for s in en_sentences])
tgt_data = torch.stack([zh_vocab.encode(s, MAX_ZH, chinese=True) for s in zh_sentences])
print(f'Vocab sizes — EN: {len(en_vocab.w2i)}, ZH: {len(zh_vocab.w2i)}')
print(f'Src shape: {src_data.shape}, Tgt shape: {tgt_data.shape}')


In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────
toy_model = Transformer(
    src_vocab=len(en_vocab.w2i), tgt_vocab=len(zh_vocab.w2i),
    d_model=64, num_heads=4, num_layers=2, d_ff=128, dropout=0.1
).to(device)

optimizer = optim.Adam(toy_model.parameters(), lr=1e-3, betas=(0.9, 0.98))
criterion = nn.CrossEntropyLoss(ignore_index=Vocab.PAD)

src = src_data.to(device)
tgt_in  = tgt_data[:, :-1].to(device)
tgt_out = tgt_data[:, 1:].to(device)

losses = []
EPOCHS = 300
for ep in range(1, EPOCHS+1):
    toy_model.train()
    mask = make_causal_mask(tgt_in.size(1), device)
    logits = toy_model(src, tgt_in, tgt_mask=mask)
    loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(loss.item())
    if ep % 50 == 0:
        print(f'Epoch {ep:3d} | Loss {loss.item():.4f}')


---
## 5. Results & Evaluation


In [ ]:
def translate(model, sentence, en_v, zh_v, max_len=10):
    model.eval()
    src = en_v.encode(sentence, MAX_EN).unsqueeze(0).to(device)
    memory = model.encode(src)
    tgt = torch.tensor([[Vocab.BOS]], device=device)
    result = []
    with torch.no_grad():
        for _ in range(max_len):
            mask = make_causal_mask(tgt.size(1), device)
            out  = model.decode(tgt, memory, tgt_mask=mask)
            next_tok = model.out_proj(out[:, -1]).argmax(-1).unsqueeze(0)
            if next_tok.item() == Vocab.EOS: break
            tgt = torch.cat([tgt, next_tok], dim=1)
            result.append(zh_v.i2w.get(next_tok.item(), '?'))
    return ''.join(result)

print('=== Translation Results ===')
for s in en_sentences[:5]:
    print(f'  EN: {s}')
    print(f'  ZH: {translate(toy_model, s, en_vocab, zh_vocab)}')
    print()


---
## 6. Visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(losses, color='steelblue', lw=1.5)
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss'); axes[0].grid(alpha=0.3)

# Positional encoding heatmap
pe_module = PositionalEncoding(d_model=64, dropout=0, max_len=50)
dummy = torch.zeros(1, 50, 64)
with torch.no_grad():
    pe_vals = pe_module.pe[0].numpy()
im = axes[1].imshow(pe_vals.T[:32], aspect='auto', cmap='RdBu', origin='lower')
axes[1].set_title('Positional Encoding (first 32 dims)')
axes[1].set_xlabel('Position'); axes[1].set_ylabel('Encoding Dimension')
plt.colorbar(im, ax=axes[1])

plt.suptitle('Transformer — Attention Is All You Need (Vaswani et al., 2017)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
